In [1]:
import os
import glob
from time import sleep
from time import perf_counter as time
from tqdm import tqdm
import numpy as np
import torch
import monai.transforms as mt

from data.ZarrIterableDatasetBase import ZarrIterableDataset

In [2]:

# Example usage of OME-Zarr dataloader using the VoDaSuRe dataset

batch_size = 4
up_factor = 2  # SR scale, supports 2x and 4x
patch_shape = [32, 32, 32]  # lo-res patch size
patch_shape_hr = [shape * up_factor for shape in patch_shape]  # hi-res patch size

seed = 8883
num_samples = 1000
num_workers = 4

base_path = "../../3D_datasets/datasets/VoDaSuRe/ome" # "<path/to/VoDaSuRe/ome>"  # <-- Set path to where ome/ folder is located

train_paths = glob.glob(os.path.join(base_path, "train", "*.zarr"))
# test_paths = glob.glob(os.path.join(base_path, "test", "*.zarr")) Repeat same code for test paths

dataset_dict = {
    "VoDaSuRe": {
        "paths": train_paths,
        "group_pairs": {
            "4": [{"H": "HR/0", "L": "REG/0"}],  # For downsampled LR input, set to {"H": "HR/0", "L": "HR/2"}
            "2": [{"H": "HR/1", "L": "REG/0"}],  # For downsampled LR input, set to {"H": "HR/0", "L": "HR/1"}
        },
        "sampling_weight": 1,
        "store_type": "LocalStore"
    }
}

torch.manual_seed(seed)
np.random.seed(seed)

# Define patch transforms
patch_transform = mt.Compose([
    mt.Identityd(keys=['H', 'L'], allow_missing_keys=True),
    mt.EnsureChannelFirstd(keys=['H', 'L'], channel_dim='no_channel'),
    mt.CastToTyped(keys=['H', 'L'], dtype=np.float32),
])

dataset = ZarrIterableDataset(dataset_dict,
                              patch_shape,
                              patch_shape_hr,
                              patch_transform,
                              up_factor=up_factor,
                              store_type='LocalStore',
                              num_samples=num_samples,
                              sampling_method='random',  # 'random' or 'in_chunk'
                              print_metadata=False,
                              slice_dim=None)

dataloader = torch.utils.data.DataLoader(dataset,
                                        batch_size=batch_size,
                                        shuffle=False,
                                        num_workers=num_workers,
                                        pin_memory=False,
                                        persistent_workers=True if num_workers > 0 else False,
                                        prefetch_factor=32)

no_epochs = 10
start_time = time()
for i in range(no_epochs):
    print(f"Epoch {i + 1}/{no_epochs}")
    for batch in tqdm(dataloader, mininterval=2):
        sleep(0.01)  # Assuming some processing time

print(f"Loaded {num_samples * no_epochs} patch pairs from Zarr dataset.")

Epoch 1/10


252it [00:35,  7.12it/s]                         


Epoch 2/10


252it [00:25,  9.82it/s]                         


Epoch 3/10


...

Epoch 9/10


252it [00:25,  9.82it/s]                         


Epoch 10/10


252it [00:25,  9.81it/s]                         

Loaded 1000 items from Zarr dataset.
